In [12]:
from google.colab import drive
# Tenta desmontar o Drive para garantir uma nova conexão limpa
try:
    drive.flush_and_unmount()
except:
    pass
# Monta o Drive novamente. Você deve ver um link e pedir a permissão.
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
from google.colab import drive
import numpy as np
import re

# ====================================================================
# 1. Montagem, Caminhos e Limpeza
# ====================================================================

print("Montando Google Drive...")
if not os.path.exists('/content/drive'):
    # Assumindo que o drive.mount foi executado com sucesso em uma célula anterior
    pass
else:
    print("Drive já montado.")

PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

# --- Solicita o nome do arquivo de entrada ---
NOME_ARQUIVO_ENTRADA = input("Digite o nome do arquivo TXT ou CSV para análise (Ex: PicoW.txt, S1.csv): ").strip()

if '.' not in NOME_ARQUIVO_ENTRADA:
    NOME_ARQUIVO_ENTRADA += '.txt'

ARQUIVO_ENTRADA = NOME_ARQUIVO_ENTRADA
CAMINHO_SAVE_ESTRUTURADO = os.path.join(PASTA_DRIVE, 'dados_estruturados.csv')
CAMINHO_COMPLETO_ENTRADA = os.path.join(PASTA_DRIVE, ARQUIVO_ENTRADA)

NOME_LOG = ARQUIVO_ENTRADA.rsplit('.', 1)[0].upper()

print(f"Caminho do arquivo de entrada definido: {CAMINHO_COMPLETO_ENTRADA}")
print(f"Caminho de saída estruturado definido: {CAMINHO_SAVE_ESTRUTURADO}")

# --- LIMPEZA DE RESULTADOS ANTERIORES ---
if os.path.exists(CAMINHO_SAVE_ESTRUTURADO):
    try:
        os.remove(CAMINHO_SAVE_ESTRUTURADO)
        print(f"✅ Arquivo de resultados anteriores removido: {os.path.basename(CAMINHO_SAVE_ESTRUTURADO)}")
    except Exception as e:
        print(f"AVISO: Não foi possível remover o arquivo de resultados anteriores. {e}")
print("-" * 50)


# ====================================================================
# 2. FUNÇÕES DE ESTRUTURAÇÃO
# ====================================================================

def processar_dataframe(df, nome_log, caminho_saida):
    """Função auxiliar para limpeza e salvamento final do DataFrame."""
    if 'Dispositivo' not in df.columns:
        df['Dispositivo'] = nome_log

    # CORREÇÃO CRÍTICA: Se a coluna Dispositivo foi atribuída pelo nome do arquivo,
    # ela não deve ser limpa logo no início, apenas o Timestamp.
    df.dropna(subset=['Timestamp'], inplace=True)

    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza inicial (Timestamp/Dispositivo).")
        return False

    # Remove qualquer linha onde o Dispositivo foi atribuído, mas é um valor nulo/vazio.
    df['Dispositivo'].replace('', np.nan, inplace=True)
    df.dropna(subset=['Dispositivo'], inplace=True)

    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza final de Dispositivo.")
        return False

    df.set_index('Timestamp', inplace=True)

    df['Temperatura'] = pd.to_numeric(df['Temperatura'], errors='coerce')
    df['Umidade'] = pd.to_numeric(df['Umidade'], errors='coerce')
    df.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza de Temperatura/Umidade.")
        return False

    # Salvar
    if os.path.exists(caminho_saida):
        df[['Dispositivo', 'Temperatura', 'Umidade']].to_csv(caminho_saida, mode='a', header=False)
    else:
        df[['Dispositivo', 'Temperatura', 'Umidade']].to_csv(caminho_saida)

    print(f"Estruturação de {nome_log} concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

def estruturar_log_regex(caminho_entrada, caminho_saida, nome_log):
    # ... (código REGEX omitido por ser irrelevante para o problema CSV)
    return False


def estruturar_log_csv(caminho_entrada, caminho_saida, nome_log):
    """
    Tenta estruturar focando no formato CSV/Tabular, com prioridade para a leitura bruta
    identificada na Imagem 1 (vírgula, sem cabeçalho).
    """
    print(f"\nTentando estruturação (CSV/Tabular) de: {os.path.basename(caminho_entrada)}")

    df = None

    # NOVAS COLUNAS BASEADAS NA IMAGEM 1 (Contagem, Data_Hora, Temperatura, Umidade)
    COLUNAS_BRUTAS_IMAGEM1 = ['Contagem_Col', 'Data_Hora_Col', 'Temperatura_Col', 'Umidade_Col']

    # Delimitadores de alta prioridade (vírgula e ponto-e-vírgula)
    separadores = [(',', 'CSV (,)'), (';', 'CSV (;)')]

    for sep_char, sep_nome in separadores:
        try:
            # === TENTATIVA 1: Leitura Bruta da Imagem 1 (sem cabeçalho e sem pular linhas) ===
            df_bruto = pd.read_csv(caminho_entrada, sep=sep_char, header=None,
                                    names=COLUNAS_BRUTAS_IMAGEM1,
                                    encoding='latin-1', engine='python', on_bad_lines='skip',
                                    skiprows=0) # NÃO PULA NENHUMA LINHA

            if df_bruto.shape[1] >= 4 and df_bruto.shape[0] > 0:
                df = df_bruto.iloc[:, :4].copy()
                print(f"INFO: Usando formato delimitado ({sep_nome}) com leitura bruta da Imagem 1 para {nome_log}")

                # 1. Renomear/Mapear as colunas internas
                df.rename(columns={
                    'Data_Hora_Col': 'Timestamp',
                    'Temperatura_Col': 'Temperatura',
                    'Umidade_Col': 'Umidade',
                }, inplace=True)

                # 2. Limpeza Extrema e Conversão de Data/Hora (REMOÇÃO DO FORMATO HARDCODED)
                # Tenta limpar a vírgula e remove espaços extras
                df['Timestamp'] = df['Timestamp'].astype(str).str.replace(',', ' ', regex=False).str.strip()

                # Tenta conversão flexível (deixando o Pandas adivinhar, mais tolerante)
                df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

                # 3. Define o Dispositivo (Fixado aqui, mas limpo em processar_dataframe)
                df['Dispositivo'] = nome_log

                # 4. Limpa colunas que não são necessárias no DataFrame final
                df = df[['Dispositivo', 'Timestamp', 'Temperatura', 'Umidade']].copy()

                # Sucesso! Processa e sai.
                if not df.dropna(subset=['Timestamp']).empty:
                    return processar_dataframe(df, nome_log, caminho_saida)


            # === TENTATIVA 2: Leitura com Cabeçalho (para o formato da Imagem 2) ===
            # Isso é necessário se o arquivo for outro (não o RASP_DHT22_04_10.CSV)

            df_teste = pd.read_csv(caminho_entrada, sep=sep_char,
                                    engine='python', on_bad_lines='skip',
                                    encoding='latin-1')

            # ... (Lógica de mapeamento da Imagem 2 omitida por ser um fallback e não a solução do problema atual)

            colunas_presentes = set(df_teste.columns)
            temp_match = 'Temperatura' in colunas_presentes or 'Temperatura(C)' in colunas_presentes
            umid_match = 'Umidade_%' in colunas_presentes or 'Umidade(%)' in colunas_presentes

            if 'Data_Hora_Coleta' in colunas_presentes and temp_match and umid_match:
                df = df_teste.copy()
                print(f"INFO: Usando formato delimitado ({sep_nome}) com cabeçalho (Formato da Imagem 2) para {nome_log}")

                df.rename(columns={
                    'Data_Hora_Coleta': 'Timestamp',
                    'Temperatura(C)': 'Temperatura',
                    'Umidade(%)': 'Umidade',
                    'Umidade_%': 'Umidade'
                }, inplace=True)

                df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M', errors='coerce')

                if 'Dispositivo' not in df.columns:
                    df['Dispositivo'] = nome_log

                df = df[['Dispositivo', 'Timestamp', 'Temperatura', 'Umidade']].copy()
                return processar_dataframe(df, nome_log, caminho_saida)


        except Exception as e:
            # print(f"Tentativa com {sep_nome} falhou: {e}")
            continue

    print(f"AVISO TABULAR/CSV: O arquivo não pôde ser lido corretamente com nenhum delimitador/codificação. Verifique a estrutura do arquivo.")
    return False


# --- EXECUÇÃO DA ESTRUTURAÇÃO (TENTATIVA MÚLTIPLA) ---

if not estruturar_log_regex(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG):
    estruturar_log_csv(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG)


# ====================================================================
# 3. Carregamento Final
# ====================================================================

df_dados = pd.DataFrame()

try:
    df_dados = pd.read_csv(CAMINHO_SAVE_ESTRUTURADO, index_col=0, parse_dates=True)
    df_dados.rename(columns={'Temperatura': 'Temp', 'Umidade': 'Umid'}, inplace=True)
    df_dados['Temp'] = pd.to_numeric(df_dados['Temp'], errors='coerce')
    df_dados['Umid'] = pd.to_numeric(df_dados['Umid'], errors='coerce')
    df_dados.dropna(subset=['Temp', 'Umid'], inplace=True)

except FileNotFoundError:
    print("\nERRO FATAL: O arquivo estruturado não foi encontrado. A estruturação falhou.")
    exit()
except Exception as e:
    print(f"\nERRO: Falha ao carregar o arquivo estruturado. {e}")
    pass

if df_dados.empty:
    print(f"\nERRO FATAL: O DataFrame {NOME_LOG} está vazio. Verifique o formato do arquivo de entrada e os separadores usados.")
    exit()
else:
    print(f"\nDataFrame {NOME_LOG} carregado com sucesso. Iniciando Análise.")

# ====================================================================
# 4. Cálculo das Médias ARITMÉTICAS
# ====================================================================

resultados_analise = []
grupos_dispositivo = df_dados.groupby('Dispositivo')

print("\n--- ⏳ Calculando Médias ARITMÉTICAS (Simples) ---")

for dispositivo, df_grupo in grupos_dispositivo:
    if len(df_grupo) < 1:
        continue

    media_temp = df_grupo['Temp'].mean()
    media_umid = df_grupo['Umid'].mean()

    resultados_analise.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Aritmetica': media_temp,
        'Umid_Média_Aritmetica': media_umid,
        'Total_Pontos': len(df_grupo)
    })

df_resultados = pd.DataFrame(resultados_analise).set_index('Dispositivo')
df_resultados.dropna(inplace=True)


# ====================================================================
# 5. Apresentação dos Resultados
# ====================================================================

# 1. Cálculo da Média Aritmética GERAL
media_temp_geral = df_dados['Temp'].mean()
media_umid_geral = df_dados['Umid'].mean()

# 2. Apresentação do Resultado FINAL (Geral)
print("\n" + "="*90)
print(f"|   ANÁLISE FINAL: MÉDIA ARITMÉTICA GERAL do arquivo: {ARQUIVO_ENTRADA}      |")
print("="*90)

print("\n--- 📊 Média ARITMÉTICA por Dispositivo (Individual) ---")
print(df_resultados[['Temp_Média_Aritmetica', 'Umid_Média_Aritmetica', 'Total_Pontos']].to_markdown(floatfmt=".2f"))


print("\n--- 🌟 Média ARITMÉTICA GERAL do Arquivo ---")
print(f"|   Temperatura Média GERAL: {media_temp_geral:.2f}°C")
print(f"|   Umidade Média GERAL:     {media_umid_geral:.2f}%")
print("------------------------------------------")

Montando Google Drive...
Drive já montado.
